# <center> Your Query Whisperer </center>

<font color='red'>**Important**: </font>
- **If the kernel doesn't attach automatically, or if you need to restart the kernel, please choose 'conda_python3' kernel.**
- Run each block of code one by one (starting from the top) either by clicking on "Run" on the menu above or by pressing SHIFT+ENTER on your keyboard.
- For the blocks that you need to complete, you see **"YOUR TURN"** in the title.
- Don't alter the code in places without ``YOUR TURN`` in the title. Just run them. 
- Have fun!

## Overview

Here is what you are going to do in this challenge:

 

   *[Task 1 : Find Latest Sales Revenue](#find-latest-sales-revenue)
   
   *[Task 2 : Data Reconciliation](#data-reconciliation)
   
   *[Task 3 : Calculate Revenue Growth Rate](#calculate-revenue-growth-rate)



<a id='Set-Up'> </a>

### Set Up

<font color='red'> Important: Please make sure to enable `Nova Lite` model access in Amazon Bedrock Console, as the notebook will use these models for completing the task.</font>

#### Follow the below steps

#### 1. Go to AWS console
#### 2. Search Amazon Bedrock in the search bar
#### 3. On the left hand side menu Bedrock configuration, Click on Model access
#### 4. Click on the Enable specific models
<img style="text-align:left"/>
  <img src="https://aws-jam-challenge-resources.s3.amazonaws.com/your-query-whisperer/1_Model_Access_Enable.png" alt="drawing" width="1000" />

#### 5. Select the model mentioned above (Nova Lite)
<img style="text-align:left"/>
  <img src="https://aws-jam-challenge-resources.s3.amazonaws.com/your-query-whisperer/3_Model_Access_Select_Models.png" alt="drawing" width="1000" />
  
#### 6. Click on Next
#### 7. Click on submit
<img style="text-align:left"/>
  <img src="https://aws-jam-challenge-resources.s3.amazonaws.com/your-query-whisperer/4_Model_Access_Submit.png" alt="drawing" width="1000" />


***
Before executing the notebook, there are some initial steps required for set up. 
***

In [1]:
import pprint
pp = pprint.PrettyPrinter(indent=2)

In [2]:
import logging
import boto3
import json
from botocore.exceptions import ClientError
import os
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


def generate_message(input_text,
                     input_document="", messages_buffer=[]):

    bedrock_client = boto3.client(service_name="bedrock-runtime")
    
    model_id = "amazon.nova-lite-v1:0"
    logger.info("Generating message with model %s", model_id)
    
    
    
    message = {
        "role": "user",
        "content": [
            {
                "text": input_text
            }
           
        ]
    }
    
    if input_document != "":

        with input_document as f:
            document = f.read()
        message = {
            "role": "user",
            "content": [
                {
                    "text": input_text
                },
                {
                    "document": {
                        "name": "MyDocument",
                        "format": "txt",
                        "source": {
                            "bytes": document
                        }
                    }
                }
            ]
        } 
    
    
    messages = []
    
    if len(messages_buffer) >= 1:
        messages = messages_buffer.copy()
        messages.append(message)
    else:
        messages = [message]
        


    response = bedrock_client.converse(
        
        system=[{"text" : "You are a financial analyst with a keen sense of attention to detail"}],
        modelId=model_id,
        messages=messages
    )
    

    all_messages = messages.copy()
    all_messages.append(response['output']['message'])

    
    return all_messages

In [3]:
s3_client = boto3.client('s3')
s3_buckets = s3_client.list_buckets()
for bucket in s3_buckets['Buckets']:
    if 'querywhisperer' in bucket['Name'].lower():
        bucket_name = bucket['Name']
        break
print(f'The S3 bucket associated with this challenge is: {bucket_name}')

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


The S3 bucket associated with this challenge is: labstack-prewarm-9b40dfd7-1cb-querywhispererbucket-mwj1azibra9j


### Define Helper Functions (please don't modify)

In [4]:
def submit_your_answer(task_file_name, task_response):
    '''Upload your response to S3 in json format. This acts as your submission!
       Arguments:
           task_file_name (str): name of the task file you are submitting 
           task_response (Dictionary): Response dictionary 
    ''' 
    task_json = json.dumps(task_response)
    with open(task_file_name, 'w') as f:
        f.write(task_json)
    s3_client.upload_file(Bucket=bucket_name, Filename=os.path.join('./', task_file_name), Key=task_file_name)


In [5]:
s3_client = boto3.client('s3')
s3_buckets = s3_client.list_buckets()

for bucket in s3_buckets['Buckets']:
    if 'querywhisperer' in bucket['Name'].lower():
        bucket_name = bucket['Name']
        break


 <a id='find-latest-sales-revenue'> </a>
### <font color='17C822'> YOUR TURN: Task 1 : Find Latest Sales Revenue </font>
You are a data analyst. You are analyzing financial statement named "fin_statements.pdf".Write a prompt to find the latest revenue from the report

In [6]:
s3 = boto3.resource('s3')
key ="fin_statements.pdf"
obj = s3.Object(bucket_name, key)
input_document = obj.get()['Body']

response = generate_message("What is octank's latest revenue", input_document)
response[1]['content']

INFO:__main__:Generating message with model amazon.nova-lite-v1:0


[{'text': "Based on the provided financial statements for Octank, the latest revenue for the year ended December 31, 2023, is **$20,000,000**.\n\nHere's a summary of the revenue for the last five years:\n\n- **2023:** $20,000,000\n- **2022:** $18,500,000\n- **2021:** $17,000,000\n- **2020:** $16,200,000\n- **2019:** $15,500,000\n\nThis indicates a steady growth in revenue over the past five years."}]

In [7]:
#Enter the revenue from above output in  $XX,XXX,XXX format
#Example :- if answer is 85000000, enter the value $85,000,000
latest_revenue =''

##################################################################################################################
### SUBMISSION (DON'T CHANGE THE CODE)
Run the following cell to submit your response. <font color='red'> **DON'T CHANGE THIS LINE** </font>. Just run it as it is once you have completed the ``latest_revenue`` dictionary above. This is your submission for Task #1. You'll receive credit shortly if you answer is correct.
##################################################################################################################

In [8]:
# DON'T CHANGE THIS LINE OR YOUR SUBMISSION FAILS
submit_your_answer('task1.json', latest_revenue)

<a id='data-reconciliation'> </a>
### <font color='17C822'> YOUR TURN: Task 2 : Data Reconciliation </font>
You are a finance head. You are analyzing financial statement named "fin_statements.csv". Write a prompt to find the latest revenue from the report and reconcile the value with the value you have got fin_statements.pdf in task 1.

In [9]:
s3 = boto3.resource('s3')
key ="fin_statements.csv"
obj = s3.Object(bucket_name, key)
input_document = obj.get()['Body']

response_csv = generate_message("What is octank's latest revenue?", input_document)
response_csv[1]['content']

INFO:__main__:Generating message with model amazon.nova-lite-v1:0


[{'text': 'Based on the provided Profit and Loss Statement for Octank, the latest revenue for the year ended December 31, 2023, is **$20,011,000**.'}]

In [10]:
#Enter the  revenue difference between CSV and PDF reports
#Enter the answer in $XX,XXX format
#Example :- if answer is 50000, enter the value $50,000
revenue_diff = ''

##################################################################################################################
### SUBMISSION (DON'T CHANGE THE CODE)
Run the following cell to submit your response. <font color='red'> **DON'T CHANGE THIS LINE** </font>. Just run it as it is once you have completed the ``revenue_diff`` dictionary above. This is your submission for Task #2. You'll receive credit shortly if you answer is correct.
##################################################################################################################

In [11]:
# DON'T CHANGE THIS LINE OR YOUR SUBMISSION FAILS
submit_your_answer('task2.json', revenue_diff)

<a id='calculate-revenue-growth-rate'> </a>
### <font color='17C822'> YOUR TURN: Task 3 : Calculate Revenue Growth Rate</font>
You are a chief financial officer. You are analyzing financial statement named "fin_statements_consolidated.pdf" to understand the average year on year revenue growth. Write a prompt to find the average year on year growth from 2019 to 2023 from the report.

In [12]:
s3 = boto3.resource('s3')
key ="fin_statements_consolidated.pdf"
obj = s3.Object(bucket_name, key)
input_document = obj.get()['Body']


response_consolidated_statement = generate_message("Find the average yoy revenue growth rate between 2019-2023", input_document)

INFO:__main__:Generating message with model amazon.nova-lite-v1:0


In [13]:
pp.pprint(response_consolidated_statement[1]['content'][0]['text'])

('To calculate the average year-over-year (YoY) revenue growth rate for Octank '
 'between 2019 and 2023, we need to determine the revenue growth rate for each '
 'year and then find the average of these growth rates.\n'
 '\n'
 'Here are the revenues for each year:\n'
 '\n'
 '- 2019: $15,500,000\n'
 '- 2020: $16,200,000\n'
 '- 2021: $17,000,000\n'
 '- 2022: $18,500,000\n'
 '- 2023: $20,000,000\n'
 '\n'
 'We will calculate the YoY growth rate for each year using the formula:\n'
 '\n'
 '\\[ \\text{YoY Growth Rate} = \\left( \\frac{\\text{Revenue in Current Year} '
 '- \\text{Revenue in Previous Year}}{\\text{Revenue in Previous Year}} '
 '\\right) \\times 100 \\]\n'
 '\n'
 '### Calculations:\n'
 '\n'
 '1. **2020 YoY Growth Rate:**\n'
 '\\[ \\left( \\frac{16,200,000 - 15,500,000}{15,500,000} \\right) \\times 100 '
 '= \\left( \\frac{700,000}{15,500,000} \\right) \\times 100 = 4.52\\% \\]\n'
 '\n'
 '2. **2021 YoY Growth Rate:**\n'
 '\\[ \\left( \\frac{17,000,000 - 16,200,000}{16,200,000} \

In [ ]:
#Enter the  average year on year growth rate between between 2019-2023 from above output
#Keep the final output restricted to nearest decimal point 
#Example :- if answer 25.39%, enter the value 25.40%
growth_rate=''

##################################################################################################################
### SUBMISSION (DON'T CHANGE THE CODE)
Run the following cell to submit your response. <font color='red'> **DON'T CHANGE THIS LINE** </font>. Just run it as it is once you have completed the ``growth_rate`` dictionary above. This is your submission for Task #3. You'll receive credit shortly if you answer is correct.
##################################################################################################################

In [ ]:
# DON'T CHANGE THIS LINE OR YOUR SUBMISSION FAILS
submit_your_answer('task3.json', growth_rate)